# Part 1 — Random variable sampling

In [ ]:
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

from scipy.special import erfinv
from scipy.stats import binom, norm

from functions import functime

# Settings: font size for plot labels
matplotlib.rcParams['font.size'] = 14

Statistical analyses heavily rely on the generation of fake datasets ("pseudo-experiments"). To build these datasets, observables are generated by randomly sampling their probability distribution functions (PDFs). For simple distributions, you can find built-in functions in numpy.random and scipy.stats. For more complex distributions you will need to write your own sampling method. Here we present the two most common sampling algorithms, assuming simple 1D distributions. We denote the random variable by $x$.

- **Rejection sampling**: generate a value of $x$ according to a __uniform__ distribution, keep it only if 
$$\frac{\mathrm{PDF}(x)}{\mathrm{PDF}_\mathrm{max}} > r$$
where $r$ is a random number in $[0,1]$

- **Inverse transform sampling**: generate a random number $y$ in $[0,1]$ according to a __uniform__ distribution. Then, search for the value of $x$ for which $\mathrm{CDF}(x) = y$

In [ ]:
# Rejection sampling
# Basic method
@functime
def rejection_sampling_basic(pdf, pdfmax, Npoints, xrange, quiet=False):
    sampled_points = []
    while len(sampled_points) < Npoints:
        x = np.random.uniform(*xrange)
        y = pdf(x)
        r = np.random.random()
        if y/pdfmax > r:
            sampled_points.append(x)
    return np.array(sampled_points)

# Vectorized method using NumPy arrays
@functime
def rejection_sampling_vectorized(pdf, pdfmax, Npoints, xrange, quiet=False):
    sampled_points = np.array([])

    while len(sampled_points) < Npoints:
        x = np.random.uniform(*xrange, size=Npoints)
        y = pdf(x)
        r = np.random.random_sample(Npoints)
        x = x[y/pdfmax > r]
        sampled_points = np.concatenate([sampled_points, x])

    return sampled_points[:Npoints]

# Inverse transform sampling
@functime
def cdf_sampling(inverse_cdf, Npoints, quiet=False):
    y = np.random.random_sample(Npoints)
    x = inverse_cdf(y)
    return x

**Python question:** What is the @functime for?

Other possible sampling methods:

- **Ziggurat sampling**: divide a monotonically decreasing distribution into N regions in y (N-1 rectangular regions + tail) and use rejection sampling inside each region. Also works on normal distributions.


- **Box-Muller algorithm**: for two values $u_1,u_2$ generated according to a uniform distribution in $[0,1]$ we can build two independent, normally-distributed, random variables:
$$x_0 = \sqrt{-2\log u_1}\cos(2\pi u_2),\; x_1 = \sqrt{-2\log u_1}\sin(2\pi u_2)$$
Here the mean is $0$ and the standard deviation is $1$. To generalize this algorithm to a distribution of mean $\mu$ and standard deviation $\sigma$, apply the following transformation:
$$x_i \rightarrow \sigma x_i + \mu$$


- **Built-in NumPy and Scipy functions**

## Test sampling methods

We will now check the speed of the different methods. Note that there are two implementations of the rejection sampling method. Which one is more "pythonic"? Why? How will the choice of method affect the performance?

Here we will generate points according to a normal distribution with mean 0 and variance $\sigma$. We will show the results using histograms. The settings are shown below.

In [ ]:
# Settings
mu, sigma = 0, 1
Npoints = 100000

def gaussian_pdf(x):
    return np.exp(-(x-mu)**2/(2*sigma**2))/np.sqrt(2*np.pi*sigma**2)

pdfmax = 1/np.sqrt(2*np.pi*sigma**2)

def gaussian_inverse_cdf(x):
    return mu + erfinv((x-0.5)*2)*np.sqrt(2*sigma**2)

xrange = (-10, 10)
nbins = int(40/sigma) + 1
bins = np.linspace(xrange[0], xrange[1], nbins)
x = np.arange(xrange[0], xrange[1], 0.01)

We plot the distribution obtained with `rejection_sampling_basic`.

In [ ]:
points = rejection_sampling_basic(gaussian_pdf, pdfmax, Npoints, xrange)
plt.hist(points, bins=bins, label="Sampled")
plt.plot(x, gaussian_pdf(x)*Npoints*(bins[1]-bins[0]), label="Target PDF")
plt.xlabel("X")
plt.ylabel("Number of trials")
plt.legend(loc="best")
plt.show()

We plot the distribution obtained with `rejection_sampling_vectorized`. Do you see some differences?

In [ ]:
points = rejection_sampling_vectorized(gaussian_pdf, pdfmax, Npoints, xrange)
plt.hist(points, bins=bins, label="Sampled")
plt.plot(x, gaussian_pdf(x)*Npoints*(bins[1]-bins[0]), label="Target PDF")
plt.xlabel("X")
plt.ylabel("Number of trials")
plt.legend(loc="best")
plt.show()

We plot the distribution obtained with `cdf_sampling`. Do you see some differences?

In [ ]:
points = cdf_sampling(gaussian_inverse_cdf, Npoints)
plt.hist(points, bins=bins, label="Sampled")
plt.plot(x, gaussian_pdf(x)*Npoints*(bins[1]-bins[0]), label="Target PDF")
plt.xlabel("X")
plt.ylabel("Number of trials")
plt.legend(loc="best")
plt.show()

- You can try varying the width of the distribution and the number of points generated. Which method is the fastest? Why?
- The current implementation of the inverse transform sampling will not give the same results as the other methods. Why? Can you see it on the plot? If not, which parameters should you modify for the difference to be visible?
- How could you modify the implementation of the inverse transform sampling method to make it give the same results as rejection sampling?

## EXTRA: fluctuations around expectation values

- If you generate N points according to a Gaussian distribution centered on 0, how many positive values do you expect? Will you always get this result?
- What is the theoretical distribution for `Npositive`? Why? Which simpler distribution does it converge to for large N?
- If the **total** number of events was not fixed, what should be the probability distribution for fluctuations in a given histogram bin?

In [ ]:
N = 3
Ntrials = 10000

# Function to generate a set of Ntrials x N values and arrange them in an array of shape (Ntrials, N)
values = cdf_sampling(gaussian_inverse_cdf, Ntrials*N).reshape((Ntrials, N))

# Extract the number of positive values generated for each trial and plot the corresponding histogram
Npositive = (values > 0).sum(axis=1)
# Distribution of Npositive
bins = np.linspace(-0.5, 2*N+0.5, 2*N+2)
plt.hist(Npositive, bins=bins, label="Sampled")
x = np.arange(0,2*N+1)
# Theoretical distributions
plt.plot(x, binom.pmf(x, N, 0.5)*Ntrials*(bins[1]-bins[0]), 'o', label="Binomial")
plt.plot(x, norm.pdf(x, loc=N/2, scale=np.sqrt(N/4))*Ntrials*(bins[1]-bins[0]), label="Gaussian")

# "Zoom" on interesting region
# Pick sensible width for the window around N/2
window_width = 5*np.sqrt(N/2)
xmin = max(N/2 - window_width, -0.5)
xmax = N/2 + window_width
plt.xlim(xmin,xmax)
plt.xlabel(f"Number of positive values ({N/2} expected)")
plt.ylabel("Number of trials")